In [1]:
# Zelle 1: Import testen
from data_reconciliation.io.reader import read_excel
from data_reconciliation.preprocessing.filter import IQRFilter, ResidualFilter, CompositeFilter, filter_report
from data_reconciliation.reconciliation.reconcile import reconcile
from data_reconciliation.visualization.plots import plot_raw_data, plot_corrections
from data_reconciliation.visualization.save import save_figure
from pathlib import Path

print("✓ Alle Module geladen")



✓ Alle Module geladen


In [2]:
# Zelle 2: Daten einlesen
ROOT = Path().resolve().parent        # funktioniert in Notebooks aus notebooks/
data = read_excel(ROOT / "data" / "demo-daten_3.xlsx")
X, A, rho = data["X"], data["A"], data["rho"]
stream_ids         = data["stream_ids"]
stream_labels      = data["stream_labels"]   # dict {int: {klarname, nominal, einheit, typ}}
print(f"X: {X.shape}, A: {A.shape}, rho: {rho}")



X: (3000, 6), A: (2, 6), rho: [0.02  0.05  0.005 0.005 0.02  0.1  ]


In [3]:
# Zelle 3: Filtern
# f = CompositeFilter([IQRFilter(k=1.5), ResidualFilter(A, threshold=3.0)], mode="and")
# f = CompositeFilter([IQRFilter(k=1.5), ResidualFilter(A, threshold=3.0)], mode="and")
# detailed = f.fit(X).transform_detailed(X)
# mask = detailed["combined"]
# filter_report(mask, detailed)

# Nur IQR-Filter
f = IQRFilter(k=1.5)
mask = f.fit_transform(X)
filter_report(mask)
X_stat = X[mask]



---------------------------------------------
  Filterung: 3000 Zeitschritte total
  Behalten:  2774  (92.5%)
  Entfernt:  226  (7.5%)
---------------------------------------------


In [4]:
# Zelle 4: Rekonziliation
result = reconcile(X[mask], A, rho)
print(f"Mittlerer SS_res: {result['SS_res'].mean():.6f}")



Mittlerer SS_res: 0.000000


In [ ]:
# Zelle 5: Plot
%matplotlib inline
#fig = plot_raw_data(X, data["stream_ids"], mask=mask)
fig1 = plot_raw_data(X, stream_ids, mask=mask, stream_labels=stream_labels)
fig2 = plot_raw_data(X[mask], stream_ids, mask=None, stream_labels=stream_labels, title_left="Rohdaten (gefiltert) - Zeitverlauf", title_right="Rohdaten (gefiltert) - Boxplot (normiert)")
fig3 = plot_corrections(X_stat, result["X_rec"], stream_ids, stream_labels=stream_labels)

In [6]:
save_figure(fig2)

  1 vorhandene Bilddatei(en) gelöscht: C:\Users\Admin\Nextcloud-RP\997_Python-Projekte\data-reconciliation\notebooks\docs\plots
  Gespeichert: C:\Users\Admin\Nextcloud-RP\997_Python-Projekte\data-reconciliation\notebooks\docs\plots\2026-03-09_rohdaten_zeitverlauf.png  (155 KB)


'C:\\Users\\Admin\\Nextcloud-RP\\997_Python-Projekte\\data-reconciliation\\notebooks\\docs\\plots\\2026-03-09_rohdaten_zeitverlauf.png'